In [1]:
from pathlib import Path
import os, sys, json, subprocess, zipfile

IN_COLAB = Path('/content').exists()
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PERSIST = Path('/content/drive/MyDrive/Thesis/UniFashion/week8')
    INPUT_DIR = PERSIST / 'inputs'
    WORK = Path('/content/unifashion_week8')
else:
    # Kaggle: attach the two ZIPs as a private Dataset, edit this path,
    # enable Internet + GPU, then Save Version with outputs before leaving.
    INPUT_DIR = Path('/kaggle/input/unifashion-week8')
    PERSIST = Path('/kaggle/working/unifashion_week8_outputs')
    WORK = Path('/kaggle/working/unifashion_week8')
RUN_NAME = 'full_train_fp32_b2_seed42'
STOP_AFTER_EPOCH = 1  # change to 10 for all remaining epochs; keep RUN_NAME
RUN_DIR = PERSIST / 'runs' / RUN_NAME
WORK.mkdir(parents=True, exist_ok=True)
PERSIST.mkdir(parents=True, exist_ok=True)
print('Inputs:', INPUT_DIR, '\nWork:', WORK, '\nPersistent output:', RUN_DIR)


Mounted at /content/drive
Inputs: /content/drive/MyDrive/Thesis/UniFashion/week8/inputs 
Work: /content/unifashion_week8 
Persistent output: /content/drive/MyDrive/Thesis/UniFashion/week8/runs/full_train_fp32_b2_seed42


In [2]:
# Extract local copies for fast image reads. Model checkpoints remain persistent.
for name in ('unifashion_week8_code.zip', 'unifashion_week8_data.zip'):
    path = INPUT_DIR / name
    if not path.is_file():
        raise FileNotFoundError(f'Upload {name} to {INPUT_DIR} first')
    with zipfile.ZipFile(path) as archive:
        for member in archive.namelist():
            if not (WORK / member).resolve().is_relative_to(WORK.resolve()):
                raise ValueError(f'Unsafe archive member: {member}')
        archive.extractall(WORK)
    print('Extracted', name, flush=True)
os.chdir(WORK)
os.environ['HF_HOME'] = str(WORK / '.cache/huggingface')

# Persist the 5-GB pretrained assets so a new Colab session can reuse them.
cache = PERSIST / 'pretrained_assets'
cache.mkdir(parents=True, exist_ok=True)
(WORK / 'checkpoints').mkdir(exist_ok=True)
asset_link = WORK / 'checkpoints/unifashion'
if not asset_link.exists():
    asset_link.symlink_to(cache, target_is_directory=True)

def run_live(command):
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1, cwd=WORK) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        if process.wait():
            raise RuntimeError(f'Command failed with exit code {process.returncode}; see output above')


Extracted unifashion_week8_code.zip
Extracted unifashion_week8_data.zip


In [3]:
if not ((3, 10) <= sys.version_info[:2] <= (3, 12)):
    raise RuntimeError('Use a Python 3.10–3.12 runtime for the pinned UniFashion environment')
# All ML commands run in fresh processes, avoiding old imports in the notebook kernel.
run_live([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-unifashion.txt'])
run_live([sys.executable, '-B', 'scripts/check_unifashion_environment.py'])
run_live([sys.executable, '-c',
          'import torch; assert torch.cuda.is_available(), "Select a GPU runtime"; '
          'print(torch.__version__, torch.cuda.get_device_name(0)); '
          'print("GPU GiB:", torch.cuda.get_device_properties(0).total_memory/2**30)'])


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.8/126.8 kB 9.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 7.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 5.6 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 266.3/266.3 kB 18.6 MB/s eta 0:00:00
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Installing backend dependencies: started
  Installing backend dependencies: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 11.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 

In [4]:
run_live([sys.executable, '-B', '-u', 'scripts/download_unifashion.py', '--with-vision'])
run_live([sys.executable, '-B', '-m', 'pytest', 'tests/test_unifashion_sanity.py',
          'tests/test_unifashion_training.py', '-q', '--junitxml=tests_gpu_environment.xml'])
config = json.loads(Path('configs/unifashion_finetune_colab.json').read_text())
audit = json.loads(Path('data/data_audit.json').read_text())
print(json.dumps(config, indent=2))
print('Data:', audit['splits'], '\nMonitor:', audit['monitor'])
print('FP32 is intentional: the previous notebook successfully trained in this mode.')


Already present: /content/unifashion_week8/checkpoints/unifashion/pretrain/none_lora_0.pt
Already present: /content/unifashion_week8/checkpoints/unifashion/bert-base-uncased/config.json
Already present: /content/unifashion_week8/checkpoints/unifashion/bert-base-uncased/vocab.txt
Already present: /content/unifashion_week8/checkpoints/unifashion/bert-base-uncased/tokenizer_config.json
Already present: /content/unifashion_week8/checkpoints/unifashion/eva_vit_g.pth
Manifest: /content/unifashion_week8/checkpoints/unifashion/download_manifest.json
................                                                         [100%]
=============================== warnings summary ===============================
tests/test_unifashion_sanity.py::test_qformer_import_without_legacy_transformers_reexports
  /usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_n

In [ ]:
from pathlib import Path
import json
import sys

WORK = Path('/content/unifashion_week8')
RUN_DIR = Path(
    '/content/drive/MyDrive/Thesis/UniFashion/week8/'
    'runs/full_train_fp32_b2_seed42'
)

checkpoint = RUN_DIR / 'last.pt'
saved_config = RUN_DIR / 'config.json'

assert checkpoint.is_file(), f'Checkpoint not found: {checkpoint}'
config = json.loads(saved_config.read_text())
assert config['precision'] == 'fp32' and config['batch_size'] == 2

run_live([
    sys.executable, '-B', '-u',
    'scripts/train_unifashion_finetune.py',
    '--config', str(saved_config),
    '--data-dir', str(WORK / 'data'),
    '--images-dir', str(WORK / 'images'),
    '--output', str(RUN_DIR),
    '--resume', str(checkpoint),
    '--stop-after-epoch', '10',
])

Verified 10000/40151 images
Verified 20000/40151 images
Verified 30000/40151 images
Verified 40000/40151 images
Loading UniFashion + frozen EVA-G; training precision: fp32
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
Resumed 25000 updates; next epoch=3, batch=7001
{"epoch": 3, "batch": 7001, "step": 25001, "samples": 2, "lr": 1.890521386207577e-05, "loss_itc": 0.0, "loss_itm": 0.001270728069357574, "loss_ttc": 0.0, "loss": 0.001270728069357574, "grad_norm_before_clip": 0.053146738559007645, "amp_retries": 0, "loss_scale_before": 1.0, "lo

In [ ]:
RUN_DIR = PERSIST / 'runs' / RUN_NAME
command = [sys.executable, '-B', '-u', 'scripts/train_unifashion_finetune.py',
           '--data-dir', str(WORK / 'data'), '--images-dir', str(WORK / 'images'),
           '--output', str(RUN_DIR), '--stop-after-epoch', str(STOP_AFTER_EPOCH)]
if (RUN_DIR / 'last.pt').exists():
    command += ['--resume', str(RUN_DIR / 'last.pt')]
elif RUN_DIR.exists():
    raise RuntimeError('This run folder has no checkpoint. Preserve its logs and choose a new RUN_NAME.')
print(' '.join(command), flush=True)
run_live(command)


/usr/bin/python3 -B -u scripts/train_unifashion_finetune.py --data-dir /content/unifashion_week8/data --images-dir /content/unifashion_week8/images --output /content/drive/MyDrive/Thesis/UniFashion/week8/runs/full_train_fp32_b2_seed42 --stop-after-epoch 1
Verified 10000/40151 images
Verified 20000/40151 images
Verified 30000/40151 images
Verified 40000/40151 images
Loading UniFashion + frozen EVA-G; training precision: fp32
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
Initial monitor loss: {'loss_itc': 0.5373469370113355, 'loss_itm': 0.2

KeyboardInterrupt: 

In [ ]:
# May also be run while inspecting a stopped session. No fabricated future epochs.
run_live([sys.executable, '-B', 'scripts/report_unifashion_finetune.py', '--run-dir', str(RUN_DIR)])
from IPython.display import display, Image, Markdown
print((RUN_DIR / 'status.json').read_text())
if (RUN_DIR / 'training_curves.png').exists():
    display(Image(filename=str(RUN_DIR / 'training_curves.png')))
display(Markdown((RUN_DIR / 'training_report.md').read_text()))


In [ ]:
import shutil
if Path('tests_gpu_environment.xml').exists():
    shutil.copyfile('tests_gpu_environment.xml', RUN_DIR / 'tests_gpu_environment.xml')
evidence_zip = PERSIST / f'{RUN_NAME}_evidence.zip'
with zipfile.ZipFile(evidence_zip, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(RUN_DIR.iterdir()):
        if path.is_file() and path.suffix in ('.json', '.jsonl', '.csv', '.log', '.png', '.md', '.xml'):
            archive.write(path, path.name)
print('Report/log/curves bundle:', evidence_zip)
print('Resume checkpoint:', RUN_DIR / 'last.pt')
print('Weights chosen by monitor loss:', RUN_DIR / 'best_monitor.pt')
